# 01 - Extracción del Contenido del Corpus

- **TFM:** Evaluación experimental de Recursive Language Models para el análisis automatizado de literatura científica

- **Autor:** Juan Antonio Jiménez Cobo

Este notebook realiza los siguientes pasos:
1. Monta Google Drive y carga el CSV con los metadatos del corpus
2. Empareja cada fila del CSV con su PDF correspondiente (`paper_01.pdf`, `paper_02.pdf`, ...)
3. Extrae el texto de cada PDF con `pymupdf` (`fitz`)
4. Aplica una limpieza básica del texto
5. Guarda cada artículo como `paper_XX.txt` en la carpeta `corpus/raw/`
6. Genera un informe de extracción en CSV



## 0. Configuración de rutas

> **IMPOTANTE:** Modifica esta celda con tus rutas correspondientes antes de ejecutar el notebook.

In [ ]:
# ============================================================
# RUTAS — ajusta según tu estructura de Google Drive
# ============================================================

# Carpeta con los artículos en formato .pdf (paper_01.pdf ... paper_55.pdf)
PDF_DIR = "/content/drive/MyDrive/corpus/papers"

# Ruta al CSV con los metadatos del corpus (exportado mediante Rayyan)
CSV_PATH = "/content/drive/MyDrive/corpus/metadata/corpus_metadata.csv"

# Carpeta de salida para los archivos .txt con el texto extraído
RAW_DIR = "/content/drive/MyDrive/corpus/raw"

# Carpeta de salida para el informe de extracción
REPORT_DIR = "/content/drive/MyDrive/corpus/reports"

# Ruta de salida para el informe de extracción
REPORT_PATH = "/content/drive/MyDrive/corpus/reports/extraction_report.csv"

# Número total de artículos en el corpus
N_PAPERS = 55

## 1. Instalación de dependencias

In [ ]:
!pip install pymupdf tiktoken --quiet

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Imports y utilidades

In [ ]:
import os
import re
import csv
import pandas as pd
import fitz
import tiktoken
from pathlib import Path

# Crear carpetas de salida si no existe
Path(RAW_DIR).mkdir(parents=True, exist_ok=True)
Path(REPORT_DIR).mkdir(parents=True, exist_ok=True)

# Tokenizador para estimar longitud en tokens (cl100k_base = GPT-4 / text-embedding-ada-002)
enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

print("✓ Imports completados")
print(f"✓ Carpeta de salida de los textos extraídos: {RAW_DIR}")
print(f"✓ Carpeta de salida del informe de extracción: {REPORT_DIR}")

## 4. Cargar metadatos del CSV

In [ ]:
df = pd.read_csv(CSV_PATH)

# Añadir columna paper_id: fila 1 → paper_01, fila 2 → paper_02, ...
df["paper_id"] = [f"paper_{str(i+1).zfill(2)}" for i in range(len(df))]

print(f"✓ CSV cargado: {len(df)} filas")
print(f"  Columnas: {list(df.columns)}")
df.head(5)

## 5. Función de limpieza de texto

Aplica las siguientes transformaciones al texto extraído del PDF:
- Elimina guiones de separación de línea (`ex-\nample` → `example`)
- Colapsa espacios múltiples y líneas en blanco redundantes
- Elimina líneas muy cortas que suelen corresponder a cabeceras o pies de página

In [ ]:
def clean_text(raw: str) -> str:
    """
    Limpieza básica del texto extraído.
    Conservativa: solo elimina artefactos de extracción,
    sin modificar el contenido semántico.
    """
    # 1. Reunir palabras partidas por guión al final de línea
    text = re.sub(r"-(\n)\s*", "", raw)

    # 2. Eliminar líneas muy cortas (≤ 3 caracteres):
    #    números de página, letras sueltas de columnas mal extraídas
    lines = text.split("\n")
    lines = [l for l in lines if len(l.strip()) > 3]
    text = "\n".join(lines)

    # 3. Colapsar múltiples líneas en blanco a una sola
    text = re.sub(r"\n{3,}", "\n\n", text)

    # 4. Colapsar espacios múltiples
    text = re.sub(r" {2,}", " ", text)

    return text.strip()


def extract_text_from_pdf(pdf_path: str) -> tuple[str, int]:
    """
    Extrae y limpia el texto de un PDF con pymupdf.
    Devuelve (texto_limpio, num_paginas).
    Lanza excepción si el PDF no tiene texto extraíble.
    """
    doc = fitz.open(pdf_path)
    n_pages = len(doc)
    pages_text = []

    for page in doc:
        page_text = page.get_text("text")
        if page_text.strip():
            pages_text.append(page_text)

    doc.close()

    if not pages_text:
        raise ValueError("PDF sin texto extraíble (posiblemente escaneado)")

    raw_text = "\n".join(pages_text)
    clean = clean_text(raw_text)
    return clean, n_pages


print("✓ Funciones definidas")

## 6. Extracción principal

Procesa los 55 PDFs e imprime el progreso en tiempo real.

In [ ]:
report = []

for idx, row in df.iterrows():
    paper_id = row["paper_id"]
    pdf_filename = f"{paper_id}.pdf"
    pdf_path = os.path.join(PDF_DIR, pdf_filename)

    title = row.get("title", paper_id) if "title" in df.columns else paper_id
    short_title = str(title)[:60] + "..." if len(str(title)) > 60 else str(title)

    record = {
        "paper_id": paper_id,
        "title": title,
        "status": None,
        "n_pages": None,
        "n_tokens": None,
        "n_chars": None,
        "error": None,
    }

    if not os.path.exists(pdf_path):
        record["status"] = "ERROR"
        record["error"] = "PDF no encontrado"
        print(f"  ✗ {paper_id} | PDF no encontrado")
        report.append(record)
        continue

    try:
        text, n_pages = extract_text_from_pdf(pdf_path)
        n_tokens = count_tokens(text)
        n_chars = len(text)

        # Guardar .txt
        txt_path = os.path.join(RAW_DIR, f"{paper_id}.txt")
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(text)

        record["status"] = "OK"
        record["n_pages"] = n_pages
        record["n_tokens"] = n_tokens
        record["n_chars"] = n_chars

        print(f"  ✓ {paper_id} | {n_pages} págs | {n_tokens:,} tokens | {short_title}")

    except Exception as e:
        record["status"] = "ERROR"
        record["error"] = str(e)
        print(f"  ✗ {paper_id} | ERROR: {e} | {short_title}")

    report.append(record)

print("\n" + "="*60)
print(f"Procesados: {len([r for r in report if r['status'] == 'OK'])} / {N_PAPERS}")
print(f"Errores:    {len([r for r in report if r['status'] == 'ERROR'])} / {N_PAPERS}")

## 7. Guardar informe de extracción

In [ ]:
report_df = pd.DataFrame(report)
report_df.to_csv(REPORT_PATH, index=False, encoding="utf-8")

print(f"✓ Informe guardado en: {REPORT_PATH}")
print()
report_df

## 8. Estadísticas del corpus

In [ ]:
ok = report_df[report_df["status"] == "OK"]

if len(ok) > 0:
    print("=" * 50)
    print("ESTADÍSTICAS DEL CORPUS (artículos extraídos correctamente)")
    print("=" * 50)
    print(f"  Artículos procesados:     {len(ok)}")
    print(f"  Total tokens:             {ok['n_tokens'].sum():,}")
    print(f"  Media tokens/artículo:    {ok['n_tokens'].mean():,.0f}")
    print(f"  Mín tokens:               {ok['n_tokens'].min():,} ({ok.loc[ok['n_tokens'].idxmin(), 'paper_id']})")
    print(f"  Máx tokens:               {ok['n_tokens'].max():,} ({ok.loc[ok['n_tokens'].idxmax(), 'paper_id']})")
    print(f"  Mediana tokens/artículo:  {ok['n_tokens'].median():,.0f}")
    print()

    # Artículos con pocos tokens
    sospechosos = ok[ok["n_tokens"] < 500]
    if len(sospechosos) > 0:
        print("⚠ ARTÍCULOS CON MENOS DE 500 TOKENS (revisar manualmente):")
        for _, r in sospechosos.iterrows():
            print(f"   - {r['paper_id']}: {r['n_tokens']} tokens")
    else:
        print("✓ Ningún artículo sospechoso (todos > 500 tokens)")

errors = report_df[report_df["status"] == "ERROR"]
if len(errors) > 0:
    print()
    print("✗ ARTÍCULOS CON ERROR:")
    for _, r in errors.iterrows():
        print(f"   - {r['paper_id']}: {r['error']}")

## 9. Inspección manual de un artículo

Cambia `PAPER_TO_INSPECT` para revisar el texto extraído de cualquier artículo.

In [ ]:
PAPER_TO_INSPECT = "paper_01"  # cambia este valor
N_CHARS_PREVIEW = 2000              # cuántos caracteres mostrar

txt_path = os.path.join(RAW_DIR, f"{PAPER_TO_INSPECT}.txt")

if os.path.exists(txt_path):
    with open(txt_path, "r", encoding="utf-8") as f:
        content = f.read()
    print(f"--- Primeros {N_CHARS_PREVIEW} caracteres de {PAPER_TO_INSPECT} ---\n")
    print(content[:N_CHARS_PREVIEW])
    print(f"\n[...] ({len(content):,} caracteres totales)")
else:
    print(f"Archivo no encontrado: {txt_path}")